In [1]:
import pyterrier as pt
from pathlib import Path
from pyterrier.measures import RR, nDCG, MAP
from fast_forward.encoder import TASBEncoder
import torch
from fast_forward.util import Indexer
from fast_forward.util.pyterrier import FFScore
from fast_forward.util.pyterrier import FFInterpolate
print(pt.__version__)

C:\Users\lazar\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


0.12.1


Our experiment consists of four phases:
1) We only evaluate BM25 performance on the datasets (evSciFact, FiQA, TREC-COVID, NFCorpus)
2) We retrieve top-k (test several k values) and then use the pretrained neural reranker and evaluate the scores. Since the neural reranker is trained on MSMACRO we can't use it for evaluation.
3) We now check out the hybrid pipeline (interpolate BM25 with neural reranker) , so here we also test several k values and some α values
4) We need to test if the α value works across datasets (fix k value - found a good one in previous step) and tune α on one dataset (e.g Fiqa) and test it on the others.

## Create sparse indexes for BM25 for all datasets
We measure the performance of BM25 on our datasets

In [2]:
# first we import all our datasets:
# finance domain with some technical language usually for QA retrieval
fiqa = pt.get_dataset("irds:beir/fiqa")

# this dataset has structured and formal language (domain:science) and used for evidence retrieval
scifact = pt.get_dataset("irds:beir/scifact")

# arguana is a semantic dataset in the domain of debates/argumentation
arguana = pt.get_dataset("irds:beir/arguana")

# general QA dataset
nfcorpus = pt.get_dataset("irds:beir/nfcorpus")


In [3]:

fiqa_indexer = pt.IterDictIndexer(
    str(Path.cwd()),  # this will be ignored
    type=pt.index.IndexingType.MEMORY,
)
fiqa_index_ref = fiqa_indexer.index(fiqa.get_corpus_iter(), fields=["text"])


Java started (triggered by TerrierIndexer.__init__) and loaded: pyterrier.java, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]
beir/fiqa documents: 100%|█████████████████████████████████████████████████████| 57638/57638 [00:18<00:00, 3039.70it/s]


In [4]:
# index for scifact
scifact_indexer = pt.IterDictIndexer(
    str(Path.cwd()),  # this will be ignored
    type=pt.index.IndexingType.MEMORY,
)
scifact_index_ref = scifact_indexer.index(scifact.get_corpus_iter(), fields=["text"])

beir/scifact documents: 100%|████████████████████████████████████████████████████| 5183/5183 [00:02<00:00, 1963.63it/s]


In [5]:
# index for nfcorpus
nfcorpus_indexer = pt.IterDictIndexer(
    str(Path.cwd()),  # this will be ignored
    type=pt.index.IndexingType.MEMORY,
)
nfcorpus_index_ref = nfcorpus_indexer.index(nfcorpus.get_corpus_iter(),fields=["text"])

beir/nfcorpus documents: 100%|███████████████████████████████████████████████████| 3633/3633 [00:01<00:00, 2220.05it/s]


In [6]:
# index for arguana
arguana_indexer = pt.IterDictIndexer(
    str(Path.cwd()),  # this will be ignored
    type=pt.index.IndexingType.MEMORY,
    meta={"docno": 100} # it has large ids
)
arguana_index_ref = arguana_indexer.index(arguana.get_corpus_iter(),fields=["text"])

beir/arguana documents: 100%|████████████████████████████████████████████████████| 8674/8674 [00:03<00:00, 2736.56it/s]


## Phase 1 - evaluate only with BM25

In [7]:
# Phase 1 - evaluate retrieval only with BM25
# fiqa dataset
bm25_fiqa = pt.terrier.Retriever(fiqa_index_ref, wmodel="BM25")
testset_fiqa = pt.get_dataset("irds:beir/fiqa/test")
pt.Experiment(
    [bm25_fiqa],
    testset_fiqa.get_topics(),
    testset_fiqa.get_qrels(),
    eval_metrics=[RR @ 10, nDCG @ 10, MAP @ 100],
)


10:27:34.877 [main] WARN org.terrier.querying.ApplyTermPipeline -- The index has no termpipelines configuration, and no control configuration is found. Defaulting to global termpipelines configuration of 'Stopwords,PorterStemmer'. Set a termpipelines control to remove this warning.


,name,RR@10,nDCG@10,AP@100
0,TerrierRetr(BM25),0.310271,0.252589,0.20864


In [8]:
# scifact dataset
bm25_scifact = pt.terrier.Retriever(scifact_index_ref, wmodel="BM25")
testset_scifact = pt.get_dataset("irds:beir/scifact/test")
pt.Experiment(
    [bm25_scifact],
    testset_scifact.get_topics(),
    testset_scifact.get_qrels(),
    eval_metrics=[RR @ 10, nDCG @ 10, MAP @ 100],
)


10:27:48.775 [main] WARN org.terrier.querying.ApplyTermPipeline -- The index has no termpipelines configuration, and no control configuration is found. Defaulting to global termpipelines configuration of 'Stopwords,PorterStemmer'. Set a termpipelines control to remove this warning.


,name,RR@10,nDCG@10,AP@100
0,TerrierRetr(BM25),0.632427,0.672167,0.626749


In [9]:
# arguana dataset
bm25_arguana = pt.terrier.Retriever(arguana_index_ref, wmodel="BM25")
testset_arguana = pt.get_dataset("irds:beir/arguana")
pt.Experiment(
    [bm25_arguana],
    testset_arguana.get_topics(),
    testset_arguana.get_qrels(),
    eval_metrics=[RR @ 10, nDCG @ 10, MAP @ 100],
)

10:27:54.756 [main] WARN org.terrier.querying.ApplyTermPipeline -- The index has no termpipelines configuration, and no control configuration is found. Defaulting to global termpipelines configuration of 'Stopwords,PorterStemmer'. Set a termpipelines control to remove this warning.


,name,RR@10,nDCG@10,AP@100
0,TerrierRetr(BM25),0.225617,0.342442,0.236988


In [10]:
# nfcorpus dataset
bm25_nfcorpus = pt.terrier.Retriever(nfcorpus_index_ref, wmodel="BM25")
testset_nfcorpus = pt.get_dataset("irds:beir/nfcorpus/test")
pt.Experiment(
    [bm25_nfcorpus],
    testset_nfcorpus.get_topics("text"),
    testset_nfcorpus.get_qrels(),
    eval_metrics=[RR @ 10, nDCG @ 10, MAP @ 100],
)

10:28:26.434 [main] WARN org.terrier.querying.ApplyTermPipeline -- The index has no termpipelines configuration, and no control configuration is found. Defaulting to global termpipelines configuration of 'Stopwords,PorterStemmer'. Set a termpipelines control to remove this warning.


,name,RR@10,nDCG@10,AP@100
0,TerrierRetr(BM25),0.534378,0.322219,0.143582


## Phase 2 - Neural reranker

We are going to use a pretrained neural model to rerank the top-k candidates. We are going to test several candidate depths to see how the performance reacts. I will test various candidate depths here but not on all the datasets, probably 2 of them are enough to make the assumptions neccessary

In [11]:
q_encoder = d_encoder = TASBEncoder(
    device="cuda:0" if torch.cuda.is_available() else "cpu"
)

In [28]:
from fast_forward.index import OnDiskIndex, Mode
# create the empty dense indexes on disk

def create_index_onDisk(data_name):
    index_name = "ffindex_" + data_name + "_tasb.h5" 
    
    ff_index_path  = Path.cwd() / "ir_experiments" / index_name
    ff_index_path.parent.mkdir(exist_ok=True, parents=True)
    
    try:
        ff_index = OnDiskIndex(
            ff_index_path,
            query_encoder=q_encoder,
            mode=Mode.MAXP,
            max_id_length=64
        )
        return ff_index
    except Exception as e:
        return  OnDiskIndex.load(ff_index_path)
    
scifact_index=create_index_onDisk("scifact")
fiqa_index=create_index_onDisk("fiqa")
arguana_index=create_index_onDisk("arguana")
nfcorpus_index=create_index_onDisk("nfcorpus")

100%|█████████████████████████████████████████████████████████████████████████| 8674/8674 [00:00<00:00, 1776348.46it/s]


In [13]:
def docs_iter(dataset):
    for d in dataset.get_corpus_iter():
        yield {"doc_id": d["docno"], "text": d["text"]}


## **ONLY RUN THE FOLLOWING THREE LINES IF WE HAVE MADE A MISTAKE IN THE IMPLEMENTATION**

In [ ]:
#fill in scifact dense index  !!!!!! DON'T RUN THIS I HAVE CREATED THIS ALREADY !!!!!!!!!
Indexer(scifact_index, d_encoder, batch_size=8).from_dicts(docs_iter(scifact))

In [ ]:
#fill in fiqa dense index !!!!!! DON'T RUN THIS I HAVE CREATED THIS ALREADY !!!!!!!!!
Indexer(fiqa_index, d_encoder, batch_size=8).from_dicts(docs_iter(fiqa))

In [14]:
#fill in arguana dense index !!!!!! DON'T RUN THIS I HAVE CREATED THIS ALREADY !!!!!!!!!
Indexer(arguana_index, d_encoder, batch_size=8).from_dicts(docs_iter(arguana))

0it [00:00, ?it/s]
beir/arguana documents: 100%|██████████████████████████████████████████████████████| 8674/8674 [16:53<00:00,  8.56it/s]
8674it [16:53,  8.56it/s]


In [29]:
# fill in nfcorpus dense index !!!! DON'T RUN THIS I HAVE CREATED THIS ALREADY !!!!
Indexer(nfcorpus_index, d_encoder, batch_size=8).from_dicts(docs_iter(nfcorpus))

0it [00:00, ?it/s]
beir/nfcorpus documents: 100%|█████████████████████████████████████████████████████| 3633/3633 [10:18<00:00,  5.87it/s]
3633it [10:18,  5.87it/s]


In [30]:
# Load scifact index into memory
def LoadFromDisk(data_name):
    index_name = "ffindex_" + data_name + "_tasb.h5"
    index = OnDiskIndex.load(
    Path.cwd() / "ir_experiments" / index_name,
    query_encoder=q_encoder,
    mode=Mode.MAXP,
    )
    return index

    


In [31]:
# load indexes into memory
scifact_index = LoadFromDisk("scifact")
scifact_index = scifact_index.to_memory()

fiqa_index = LoadFromDisk("fiqa")
fiqa_index = fiqa_index.to_memory()

arguana_index =LoadFromDisk("arguana")
arguana_index = arguana_index.to_memory()

nfcorpus_index = LoadFromDisk("nfcorpus")
nfcorpus_index = nfcorpus_index.to_memory()

100%|█████████████████████████████████████████████████████████████████████████| 3633/3633 [00:00<00:00, 1746064.68it/s]


### Reranking BM25 results
Now that we have dealt with all the dense indexes we can use BM25 to retrieve some documents and then use the reranking pipeline

In [33]:
def twoStageCandidateDepthExperiment(dense_index,test_dataset,dataset_name,bm25,candidate_depths,alpha_value=0.5):
    
    ff_score = FFScore(dense_index)
    ff_int = FFInterpolate(alpha=alpha_value) # we used a fixed alpha here maybe this needs to change
    pipelines = [bm25]
    names = ["BM25"]
    
    for k in candidate_depths:
        # generate the two stage pipe
        # bm25 retrieves top-k then tasb rescores them then we use interpolation to combine scores
        pipe = (bm25%k) >> ff_score >> ff_int
        pipelines.append(pipe)
        names.append(f"BM25 >> TAS-B (k={k})")
    print(f"Results for {dataset_name} ")
    results = pt.Experiment(
        pipelines,
        test_dataset.get_topics("text"),
        test_dataset.get_qrels(),
        eval_metrics=[RR @ 10, nDCG @ 10, MAP @ 100],
        names=names,
    )
    print(results)

# this list will be used to find the best k for each of the datasets
candidate_depths = [10,50,100,200,300,400,500,800,1000]

In [24]:
# run experiment on any dataset I want
twoStageCandidateDepthExperiment(scifact_index, testset_scifact, "SciFact", bm25_scifact,candidate_depths)

Results for SciFact 
                     name     RR@10   nDCG@10    AP@100
0                    BM25  0.632427  0.672167  0.626749
1    BM25 >> TAS-B (k=10)  0.650517  0.685043  0.637451
2    BM25 >> TAS-B (k=50)  0.651521  0.689003  0.643959
3   BM25 >> TAS-B (k=100)  0.650354  0.687411  0.644392
4   BM25 >> TAS-B (k=200)  0.650831  0.688523  0.644888
5   BM25 >> TAS-B (k=300)  0.650831  0.688523  0.644836
6   BM25 >> TAS-B (k=400)  0.650831  0.688523  0.644933
7   BM25 >> TAS-B (k=500)  0.650831  0.688523  0.645123
8   BM25 >> TAS-B (k=800)  0.650831  0.688523  0.645123
9  BM25 >> TAS-B (k=1000)  0.650831  0.688523  0.645123


### Interpretation and best k - SciFact
A key result I can see from the neural reranking while adjusting the candidate depth for scifact is that performance saturates somewhere around 100-200 candidate depth (nDCG@10 and MAP@100 both go flat around 200). Increasing candidate depth beyond this depth doesn't yield any improvements. To balance efficiency and effectiveness , the **best k for SciFact is k=100**

In [21]:
twoStageCandidateDepthExperiment(fiqa_index, testset_fiqa, "FIQA", bm25_fiqa,candidate_depths)

Results for fiqa 
                     name     RR@10   nDCG@10    AP@100
0                    BM25  0.310271  0.252589  0.208640
1    BM25 >> TAS-B (k=10)  0.349528  0.274255  0.219525
2    BM25 >> TAS-B (k=50)  0.367266  0.302927  0.245519
3   BM25 >> TAS-B (k=100)  0.368838  0.304480  0.250095
4   BM25 >> TAS-B (k=200)  0.369985  0.306637  0.252817
5   BM25 >> TAS-B (k=300)  0.369933  0.306817  0.253445
6   BM25 >> TAS-B (k=400)  0.370282  0.307112  0.253751
7   BM25 >> TAS-B (k=500)  0.371054  0.308301  0.254914
8   BM25 >> TAS-B (k=800)  0.371311  0.308638  0.255154
9  BM25 >> TAS-B (k=1000)  0.370925  0.308329  0.254810


### Interpretation and best k - FIQA
Although the strictly best k is k = 800 (nDCG@10 and MAP@100 highest values), going from k=1 to k=200 the gains are significant, however going from 200-1000 the gain is almost flat. Especially if I inspect the range from 500 to 1000 there is marginal gain. So taking into consideration the Efficiency vs Effectiveness tradeoff we believe that **k=200 , for FIQA** is the best for near-optimal performance

In [19]:
# !!!!!!!!!!!!!!!!!!!!!!DON'T RUN THIS AGAIN IT TAKES FAR TOO LONG!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
twoStageCandidateDepthExperiment(arguana_index,testset_arguana,"arguana",bm25_arguana,candidate_depths)

Results for arguana 
                     name     RR@10   nDCG@10    AP@100
0                    BM25  0.225617  0.342442  0.236988
1    BM25 >> TAS-B (k=10)  0.232441  0.347960  0.232026
2    BM25 >> TAS-B (k=50)  0.235005  0.353437  0.245761
3   BM25 >> TAS-B (k=100)  0.235023  0.353456  0.246529
4   BM25 >> TAS-B (k=200)  0.235125  0.353693  0.246860
5   BM25 >> TAS-B (k=300)  0.235125  0.353693  0.246844
6   BM25 >> TAS-B (k=400)  0.235125  0.353693  0.246848
7   BM25 >> TAS-B (k=500)  0.235125  0.353693  0.246857
8   BM25 >> TAS-B (k=800)  0.235125  0.353693  0.246864
9  BM25 >> TAS-B (k=1000)  0.235125  0.353693  0.246873


### Interpretation and best k - ArguAna
The results here are clear, after k=200 we see marginal if any improvement in nDCG@10 and MAP@100. It is worth noting that going deep in candidate depth here is very expensive and takes a lot of time due to the structure of the dataset. Taking this into consideration and the fact that performance saturates between k=100 and k=200 , we believe that the **best k for ArguAna is k=100**


In [34]:
# !!!!!!!!!!!!!!!!!!!!!!DON'T RUN THIS AGAIN IT TAKES FAR TOO LONG!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
twoStageCandidateDepthExperiment(nfcorpus_index,testset_nfcorpus,"NFCorpus",bm25_nfcorpus,candidate_depths)

Results for NFCorpus 
                     name     RR@10   nDCG@10    AP@100
0                    BM25  0.534378  0.322219  0.143582
1    BM25 >> TAS-B (k=10)  0.535859  0.323960  0.120096
2    BM25 >> TAS-B (k=50)  0.542887  0.337396  0.145438
3   BM25 >> TAS-B (k=100)  0.543497  0.339343  0.150670
4   BM25 >> TAS-B (k=200)  0.543110  0.339590  0.152021
5   BM25 >> TAS-B (k=300)  0.543110  0.339796  0.151994
6   BM25 >> TAS-B (k=400)  0.543110  0.339539  0.152180
7   BM25 >> TAS-B (k=500)  0.543110  0.339539  0.152182
8   BM25 >> TAS-B (k=800)  0.543110  0.339539  0.152212
9  BM25 >> TAS-B (k=1000)  0.543110  0.339539  0.152273


### Interpretation and best k - NFCorpus
The results here are clear, performance improves substantially up to around k=100 and then it saturates with marginal improvement until k=200 (we see marginal if any improvement in nDCG@10 and MAP@100). Taking into consideration effectiveness vs efficiency, we believe that the **best k for NFCorpus is k=100** (approximately, as it might be near optimal)


## Phase 3 - Test robustness of alpha parameter

Tune alpha on one dataset, and apply it to all the others


Have a list for "best alpha" per dataset. Try this alpha on all other datasets. We used a fixed candidade depth in all the datasets. After the tests in phase 2, in order to balance efficiency with effectiveness we are going to use **fixed k value = 100** , which seems to be near optimal for all the datasets tested.

In [35]:
fixed_k_value = 100

In [54]:
# we pass in the dataset to find the best alpha for the specific dataset (e.g SciFact) then we use this alpha
def findBestAlpha(bm25, ff_score,dataset, candidate_depth=100):
    alphas = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
    ff_int = FFInterpolate(alpha=0.5)
    gs = pt.GridSearch(
    (bm25 % candidate_depth) >> ff_score >> ff_int,
    {ff_int: {"alpha": alphas}},
    dataset.get_topics("text"),
    dataset.get_qrels(),
    "map",
    verbose=True,
    )
    best_alpha = ff_int.get_parameter("alpha")
    print("Best alpha is:",best_alpha)
    return best_alpha

# def evaluateBestAlpha(bm25,best_ff_pipe,testset,tested_name):

#     results = pt.Experiment(
#         [bm25, best_ff_pipe],
#         testset.get_topics("text"),
#         testset.get_qrels(),
#         eval_metrics=[RR @ 10, nDCG @ 10, MAP @ 100],
#         names=["BM25", "BM25 >> FF (tuned α)"],
#         )
#     print(results)

In [55]:
# since I want to test how well the alpha parameter generalizes I can find it upon performing the process on the dataset
# of each domain and then testing this alpha on the other three datasets.
dataset_names = ["FIQA","SciFact","ArguAna","NFCorpus"]
all_datasets = [testset_fiqa, testset_scifact, testset_arguana, testset_nfcorpus]
all_bm25 = [bm25_fiqa,bm25_scifact,bm25_arguana,bm25_nfcorpus]
all_dense_indexes = [fiqa_index,scifact_index,arguana_index,nfcorpus_index]
best_alpha_per_pipeline = []

for i in range(len(all_datasets)): # we find the optimal on each of the datasets and test it on the other (different domain) datasets
    best_alpha_per_pipeline.append(findBestAlpha(all_bm25[i],FFScore(all_dense_indexes[i]),all_datasets[i],fixed_k_value))

GridScan: 100%|██████████████████████████████████████████████████████████████████████████| 9/9 [02:43<00:00, 18.18s/it]


Best map is 0.260343
Best setting is ['<fast_forward.util.pyterrier.FFInterpolate object at 0x0000021D50FC7050> alpha=0.2']
Best alpha is: 0.2


GridScan: 100%|██████████████████████████████████████████████████████████████████████████| 9/9 [01:23<00:00,  9.33s/it]


Best map is 0.652447
Best setting is ['<fast_forward.util.pyterrier.FFInterpolate object at 0x0000021D65331860> alpha=0.3']
Best alpha is: 0.3


GridScan: 100%|█████████████████████████████████████████████████████████████████████████| 9/9 [42:23<00:00, 282.58s/it]


Best map is 0.246529
Best setting is ['<fast_forward.util.pyterrier.FFInterpolate object at 0x0000021D65331860> alpha=0.5']
Best alpha is: 0.5


GridScan: 100%|██████████████████████████████████████████████████████████████████████████| 9/9 [00:49<00:00,  5.49s/it]

Best map is 0.152672
Best setting is ['<fast_forward.util.pyterrier.FFInterpolate object at 0x0000021D652F45F0> alpha=0.4']
Best alpha is: 0.4


In [ ]:
# Grid search takes a long time so just in case something goes wrong and we lose progress:
substitute_grid_search_values = [0.2,0.3,0.5,0.4]

In [58]:
# Now best_alpha_per_pipeline can be used on each of the other datasets

for i in range(len(best_alpha_per_pipeline)):
    current_optimal = best_alpha_per_pipeline[i]
    for j in range(len(best_alpha_per_pipeline)):
        if i == j:
            continue
    # else call function that tests the result with specific alpha.
        print(f"Testing optimal alpha of {dataset_names[i]} with dataset: {dataset_names[j]}\n")
        twoStageCandidateDepthExperiment(all_dense_indexes[j],all_datasets[j],dataset_names[j],all_bm25[j],[fixed_k_value],current_optimal)
 # this also takes quite some time to get results, due to ArguAna   

Testing optimal alpha of FIQA with dataset: SciFact

Results for SciFact 
                    name     RR@10   nDCG@10    AP@100
0                   BM25  0.632427  0.672167  0.626749
1  BM25 >> TAS-B (k=100)  0.645960  0.678132  0.640733
Testing optimal alpha of FIQA with dataset: ArguAna

Results for ArguAna 
                    name     RR@10   nDCG@10    AP@100
0                   BM25  0.225617  0.342442  0.236988
1  BM25 >> TAS-B (k=100)  0.228982  0.346624  0.240296
Testing optimal alpha of FIQA with dataset: NFCorpus

Results for NFCorpus 
                    name     RR@10   nDCG@10    AP@100
0                   BM25  0.534378  0.322219  0.143582
1  BM25 >> TAS-B (k=100)  0.550047  0.334810  0.151108
Testing optimal alpha of SciFact with dataset: FIQA

Results for FIQA 
                    name     RR@10   nDCG@10    AP@100
0                   BM25  0.310271  0.252589  0.208640
1  BM25 >> TAS-B (k=100)  0.376556  0.313697  0.258812
Testing optimal alpha of SciFact with dataset

### Note 
maybe to properly test if it generalizes well we have to also check how the optimal alphas work on the same dataset or something, at least for the ones that have the test,train,dev split